## Libraries

In [25]:
import pandas as pd
import numpy as np
import spacy
import re
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
from spacy.cli import download
download("en_core_web_sm")

## Loading Dataset

In [2]:
df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")
print("First 5 records:", df.head())

First 5 records:    Unnamed: 0  Clothing ID  Age                    Title  \
0           0          767   33                      NaN   
1           1         1080   34                      NaN   
2           2         1077   60  Some major design flaws   
3           3         1049   50         My favorite buy!   
4           4          847   47         Flattering shirt   

                                         Review Text  Rating  Recommended IND  \
0  Absolutely wonderful - silky and sexy and comf...       4                1   
1  Love this dress!  it's sooo pretty.  i happene...       5                1   
2  I had such high hopes for this dress and reall...       3                0   
3  I love, love, love this jumpsuit. it's fun, fl...       5                1   
4  This shirt is very flattering to all due to th...       5                1   

   Positive Feedback Count   Division Name Department Name Class Name  
0                        0       Initmates        Intimate  Int

## Exploring Data

In [3]:
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               23486 non-null  int64 
 1   Clothing ID              23486 non-null  int64 
 2   Age                      23486 non-null  int64 
 3   Title                    19676 non-null  object
 4   Review Text              22641 non-null  object
 5   Rating                   23486 non-null  int64 
 6   Recommended IND          23486 non-null  int64 
 7   Positive Feedback Count  23486 non-null  int64 
 8   Division Name            23472 non-null  object
 9   Department Name          23472 non-null  object
 10  Class Name               23472 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.0+ MB


## Cleaning & Preprocessing

In [5]:
df = df.drop(columns=['Unnamed: 0', 'Clothing ID', 'Division Name', 'Class Name'])

In [6]:
df.isnull().sum()

Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Department Name              14
dtype: int64

In [7]:
df['Department Name'].value_counts()

Department Name
Tops        10468
Dresses      6319
Bottoms      3799
Intimate     1735
Jackets      1032
Trend         119
Name: count, dtype: int64

In [8]:
df.loc[df['Department Name'].isnull()]

,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Department Name
9444,25,My favorite socks!!!,"I never write reviews, but these socks are so ...",5,1,0,NaN
13767,23,So soft!,I just love this hoodie! it is so soft and com...,5,1,1,NaN
13768,49,Wardrobe staple,Love this hoodie. so soft and goes with everyt...,5,1,0,NaN
13787,48,NaN,NaN,5,1,0,NaN
16216,36,Warm and cozy,"Just what i was looking for. soft, cozy and warm.",5,1,0,NaN
16221,37,Love!,I am loving these. they are quite long but are...,5,1,0,NaN
16223,39,"""long and warm""",These leg warmers are perfect for me. they are...,5,1,0,NaN
18626,34,Nubby footless tights,"These are amazing quality. i agree, size up to...",5,1,5,NaN
18671,54,New workhorse,These tights are amazing! if i care for them w...,5,1,0,NaN
20088,50,Comfy sweatshirt!,This sweatshirt is really nice! it's oversize...,5,1,0,NaN


In [9]:
df = df.loc[~(df['Department Name'].isnull() | (df['Review Text'].isnull()))].copy()

In [10]:
def infer_intent(text):
    text_l = str(text).lower()
    if any(kw in text_l for kw in ['refund', 'return', 'money back']):
        return 'Refund Request'
    if any(kw in text_l for kw in ['late', 'delay', 'delivery', 'shipping']):
        return 'Delivery Issue'
    if any(kw in text_l for kw in ['complain', 'bad', 'poor', 'terrible', 'worst', 'awful', 'disappointed']):
        return 'Complaint'
    return 'General Query'

df['intent'] = df['Review Text'].apply(infer_intent)
df['intent'].value_counts()

intent
General Query     19300
Refund Request     1717
Complaint          1276
Delivery Issue      335
Name: count, dtype: int64

## Intent Labeling (Heuristic Rules)

In [11]:
def reviews(x):
  x = int(x)
  if x > 3:
    return 'positive'
  elif x == 3:
    return 'neural'
  else:
    return 'negative'

In [12]:
df['result'] = df['Rating'].apply(reviews)

In [13]:
df['result'].value_counts()

result
positive    17435
neural       2823
negative     2370
Name: count, dtype: int64

In [14]:
df['Title'] = df['Title'].str.lower()
df['Review Text'] = df['Review Text'].str.lower()

In [15]:
df['Review Text'] = df['Review Text'].str.replace(r'[^\w\s]','', regex = True)

In [16]:
nlp = spacy.load("en_core_web_sm")
def clean_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    cleaned = ' '.join(tokens)

    # Remove extra spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    return cleaned

In [17]:
clean_t = df['Review Text'].apply(clean_text)

In [18]:
sample_text = df['Review Text'].iloc[0]
print("Original Review:\n", sample_text)
print("\nAfter Cleaning:\n", clean_text(sample_text))

Original Review:
 absolutely wonderful  silky and sexy and comfortable

After Cleaning:
 absolutely wonderful silky sexy comfortable


## Preprocessing: Before and After Example

## VADER Sentiment Analysis

In [19]:
!pip install vaderSentiment

In [20]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

text = df.loc[0,'Review Text']
analyzer = SentimentIntensityAnalyzer()
analyzer.polarity_scores(text)

{'neg': 0.0, 'neu': 0.272, 'pos': 0.728, 'compound': 0.8932}

## Count Vectorizer

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english', min_df=2, token_pattern=r'[A-Za-z]+')

In [22]:
dtm = cv.fit_transform(df['Review Text'])
dtm_df = pd.DataFrame(dtm.toarray(), columns=cv.get_feature_names_out())
dtm_df

,aa,ab,abbey,abby,abdomen,ability,able,abnormally,abo,abou,...,zillion,zip,zipped,zipper,zippered,zippers,zipping,zips,zone,zoom
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,1,0,1,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22623,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
22624,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
22625,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
22626,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## TF-IDF

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(stop_words='english', min_df=2, token_pattern=r'[A-Za-z]+')
tf_idf_dtm = tv.fit_transform(clean_t)
tf_idf_dtm = pd.DataFrame(tf_idf_dtm.toarray(), columns=tv.get_feature_names_out())
tf_idf_dtm

,aa,ab,abbey,abby,abdomen,ability,able,abnormally,abo,abou,...,zig,zigzag,zillion,zip,zipped,zipper,zippered,zips,zone,zoom
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.143037,0.0,0.130796,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22623,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
22624,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
22625,0.0,0.0,0.0,0.0,0.0,0.0,0.422332,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
22626,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0


In [24]:
from sklearn.decomposition import NMF

n_topics = 5
nmf_model = NMF(n_components=n_topics, random_state=42)
W = nmf_model.fit_transform(tf_idf_dtm.values)
H = nmf_model.components_
feature_names = tv.get_feature_names_out()

def display_topics(H, feature_names, top_words=10):
    for topic_idx, topic in enumerate(H):
        top_indices = topic.argsort()[:-top_words - 1:-1]
        top_terms = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx}:", ', '.join(top_terms))

display_topics(H, feature_names, top_words=10)

Topic 0: love, great, wear, color, jean, comfortable, fit, buy, soft, perfect
Topic 1: dress, wear, love, beautiful, flattering, perfect, slip, comfortable, summer, fit
Topic 2: size, small, order, run, large, m, fit, medium, usually, big
Topic 3: look, like, fabric, nice, short, think, feel, good, material, long
Topic 4: shirt, cute, super, little, white, wear, boxy, love, underneath, tee


## Topic Modeling with NMF

In [26]:
X_intent = tf_idf_dtm
y_intent = df['intent']

X_train_intent, X_test_intent, y_train_intent, y_test_intent = train_test_split(
    X_intent, y_intent, test_size=0.2, random_state=42
)

intent_model = LogisticRegression(max_iter=1000)
intent_model.fit(X_train_intent, y_train_intent)
pred_intent = intent_model.predict(X_test_intent)

print("Intent Accuracy:", accuracy_score(y_test_intent, pred_intent))
print("\nIntent Classification Report:\n", classification_report(y_test_intent, pred_intent))
print("\nIntent Confusion Matrix:\n", confusion_matrix(y_test_intent, pred_intent))

Intent Accuracy: 0.9582412726469288

Intent Classification Report:
                 precision    recall  f1-score   support

     Complaint       0.89      0.54      0.67       244
Delivery Issue       1.00      0.16      0.28        55
 General Query       0.96      1.00      0.98      3866
Refund Request       1.00      0.94      0.97       361

      accuracy                           0.96      4526
     macro avg       0.96      0.66      0.72      4526
  weighted avg       0.96      0.96      0.95      4526


Intent Confusion Matrix:
 [[ 131    0  113    0]
 [   5    9   41    0]
 [  10    0 3856    0]
 [   1    0   19  341]]


## Intent Classification (Logistic Regression on TF-IDF)

In [ ]:
# TF-IDF Logistic Regression and comparison
X_tfidf = tf_idf_dtm
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    X_tfidf, labels, test_size=0.2, random_state=42
)

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train_tfidf)
pred_tfidf = model_tfidf.predict(X_test_tfidf)

print("TF-IDF Accuracy:", accuracy_score(y_test_tfidf, pred_tfidf))
print("\nTF-IDF Classification Report:\n", classification_report(y_test_tfidf, pred_tfidf))
print("\nTF-IDF Confusion Matrix:\n", confusion_matrix(y_test_tfidf, pred_tfidf))

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

# BoW (CountVectorizer) Logistic Regression with full metrics
model_bow = LogisticRegression(max_iter=1000)
model_bow.fit(X_train, y_train)
pred_bow = model_bow.predict(X_test)

print("BoW Accuracy:", accuracy_score(y_test, pred_bow))
print("\nBoW Classification Report:\n", classification_report(y_test, pred_bow))
print("\nBoW Confusion Matrix:\n", confusion_matrix(y_test, pred_bow))

## Logistic Regression

In [27]:
X_bow = dtm_df
labels = df['result']

X_train, X_test, y_train, y_test = train_test_split(
    X_bow, labels, test_size=0.2, random_state=42
)

In [28]:
model = LogisticRegression()
model.fit(X_train, y_train)
pred_bow = model.predict(X_test)

print("BoW Accuracy:", accuracy_score(y_test, pred_bow))

d:\SEM-6\NLP\A1\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


BoW Accuracy: 0.8009279717189571


## Gradio Interface for Customer Reviews Intelligence System

In [29]:
%pip install gradio

  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached pillow-12.1.1-cp310-cp310-win_amd64.whl.metadata (9.0 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl.metadata (2.4 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached hf_xet-1.3.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.2 MB ? eta -:--:--
   -- ------------------------------------- 1.3/24.2 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/24.2 MB 5.0 MB/s eta 0:00:05
   ----- -------------------------

In [ ]:
import os
import re

import gradio as gr
import joblib
import pandas as pd
import spacy
from spacy.cli import download
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.decomposition import NMF


# ------------------------
# Load language model
# ------------------------

download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")


# ------------------------
# Simple text cleaning
# ------------------------

def clean_text(text: str) -> str:
    """Lowercase, remove links and extra characters, then lemmatize.

    This is a simplified version of the cleaning used in preprocessing.py.
    """
    if not isinstance(text, str):
        text = str(text)

    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    doc = nlp(text)

    tokens = [
        token.lemma_
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token) > 2
    ]

    return " ".join(tokens)


# ------------------------
# Load trained models
# ------------------------

MODELS_DIR = os.path.join("..", "models")

sentiment_model_path = os.path.join(MODELS_DIR, "sentiment_model.pkl")
intent_model_path = os.path.join(MODELS_DIR, "intent_model.pkl")
tfidf_vectorizer_path = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
tfidf_dtm_path = os.path.join(MODELS_DIR, "tfidf_dtm.csv")

sentiment_model = joblib.load(sentiment_model_path)
intent_model = joblib.load(intent_model_path)
tfidf_vectorizer = joblib.load(tfidf_vectorizer_path)


# ------------------------
# Topic model (NMF)
# ------------------------

tfidf_matrix_df = pd.read_csv(tfidf_dtm_path)

n_topics = 5
nmf_model = NMF(n_components=n_topics, random_state=42)
nmf_model.fit(tfidf_matrix_df.values)

feature_names = tfidf_vectorizer.get_feature_names_out()


# ------------------------
# VADER analyzer
# ------------------------

analyzer = SentimentIntensityAnalyzer()


sentiment_label_map = {
    "positive": "Positive",
    "neutral": "Neutral",
    "negative": "Negative",
}


def analyze_review(review_text: str):
    """Analyze a single review and return sentiment, intent and topic."""

    # Rule-based sentiment (VADER)
    vader_scores = analyzer.polarity_scores(review_text)
    vader_compound = vader_scores["compound"]

    if vader_compound >= 0.05:
        vader_sentiment = "Positive"
    elif vader_compound <= -0.05:
        vader_sentiment = "Negative"
    else:
        vader_sentiment = "Neutral"

    # ML-based sentiment (LogReg + TF-IDF)
    cleaned = clean_text(review_text)
    X_new = tfidf_vectorizer.transform([cleaned])
    ml_sentiment = sentiment_model.predict(X_new)[0]
    ml_sentiment_readable = sentiment_label_map.get(ml_sentiment, ml_sentiment)

    # Intent prediction
    intent_pred = intent_model.predict(X_new)[0]

    # Topic / keywords from NMF
    topic_scores = nmf_model.transform(X_new)
    topic_idx = int(topic_scores.argmax(axis=1)[0])

    topic_components = nmf_model.components_[topic_idx]
    top_indices = topic_components.argsort()[-5:][::-1]
    top_keywords = [feature_names[i] for i in top_indices]
    topic_description = f"Topic {topic_idx}: " + ", ".join(top_keywords)

    return vader_sentiment, ml_sentiment_readable, intent_pred, topic_description


iface = gr.Interface(
    fn=analyze_review,
    inputs=gr.Textbox(lines=4, label="Enter customer review"),
    outputs=[
        gr.Textbox(label="Rule-based Sentiment (VADER)"),
        gr.Textbox(label="ML Sentiment (Logistic Regression + TF-IDF)"),
        gr.Textbox(label="Predicted Intent"),
        gr.Textbox(label="Identified Topic / Keywords"),
    ],
    title="Customer Reviews Intelligence System",
    description=(
        "Analyze customer reviews to predict sentiment, intent, and "
        "identify main topics using NLP and Machine Learning."
    ),
)


if __name__ == "__main__":
    iface.launch(share=False)

d:\SEM-6\NLP\A1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\SEM-6\NLP\A1\.venv\lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
  File "d:\SEM-6\NLP\A1\.venv\lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
  File "d:\SEM-6\NLP\A1\.venv\lib\site-packages\gradio\blocks.py", line 2157, in process_api
    result = await self.call_function(
  File "d:\SEM-6\NLP\A1\.venv\lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
  File "d:\SEM-6\NLP\A1\.venv\lib\site-packages\anyio\to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
  File "d:\SEM-6\NLP\A1\.venv\lib\site-packages\anyio\_backends\_asyncio.py", line 2502, in run_sync_in_worker_thread
    return await future
  File "d:\SEM-6\NLP\A1\.venv\lib\site-packages\anyio\_backends\_asyncio.py", 